## RAG with multiple datasources


In [ ]:
 # wikipedia tool from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=250) # it means that it will return top 5 results for the query
wiki = WikipediaQueryRun(api_wrapper=api_wrapper) # wikipediaquery run is an agent tool that allows us to query wikipedia and get the results in a structured format # AGENT TOOL
print(wiki.name)

wikipedia


In [ ]:
# basically here firstly we loaded the webpage and then we splitted the webpage into chunks of 1000 characters and then we created embeddings for those chunks 
# then we created a vector store for those chunks then we created a retriever for those chunks.
# So now we can use this retriever to retrieve the relevant chunks for a given query.
from langchain_community.document_loaders import WebBaseLoader 
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
embeddings = OpenAIEmbeddings()
vectordb = FAISS.from_documents(documents, embeddings)
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E789067110>, search_kwargs={})

In [47]:
from langchain_core.tools.retriever import create_retriever_tool # this is a function that allows us to create a retriever tool from a retriever object. 
# A retriever tool is a tool that can be used by an agent to retrieve relevant information from a retriever object.
retrieval_tool = create_retriever_tool(retriever,"wikipedia_search","Search for information about AI on wikipedia")

In [ ]:
# We also need to create our local pdf reader tool as well so that we can use it to read the pdf files and get the information from it.
local_pdf_tool = create_retriever_tool(
    retriever=vector_store.as_retriever(),
    name="local_pdf_search",
    description="Search the user's uploaded local PDFs. Always use this first if the user asks about their own documents."
)

'wikipedia_search'

In [49]:
 # Arxiv tool 
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)
print(arxiv.name)

arxiv


In [50]:
tools=[wiki, arxiv, retrieval_tool]

In [51]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
from langchain_openai import ChatOpenAI 
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0.9)

In [52]:
## Prompt Template
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the tools provided to answer questions accurately."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [53]:
from langchain.agents import create_agent

# 1. Create the Agent (This now handles the execution loop automatically)
agent = create_agent(
    model=llm, 
    tools=tools, 
    system_prompt="You are a helpful assistant. Use your tools to answer questions accurately."
)


In [58]:
# Instead of invoke, we use stream
inputs = {"messages": [("user", "What is latest page on wikipedia ??")]}

# This loop prints every "step" or "node" the agent hits
for step in agent.stream(inputs, stream_mode="values"):
    # Print the last message in the sequence
    last_message = step["messages"][-1]
    
    # This check identifies if the AI is "Thinking" or "Calling a Tool"
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print(f"\n--- 🤖 AGENT IS CALLING A TOOL: {last_message.tool_calls[0]['name']} ---")
    elif last_message.type == "human":
        pass # Skip printing the user's own question
    else:
        print(f"\n--- 💡 FINAL ANSWER ---\n{last_message.content}")


--- 🤖 AGENT IS CALLING A TOOL: wikipedia ---

--- 💡 FINAL ANSWER ---
Page: Portal:Current events
Summary: 

Page: Portal:Current events/Calendars
Summary: Here are calendars for each month since 2002 with links to archived versions of Wikipedia's Current events Portal. A red link indicates there is no archive for that

--- 💡 FINAL ANSWER ---
The latest page on Wikipedia is the "Portal:Current events/Calendars". This page contains calendars for each month since 2002 with links to archived versions of Wikipedia's Current events Portal.
